# 04 - Iceberg to ClickHouse

Notebook para revisar la tabla Iceberg, hacer un EDA corto y luego cargarla hacia ClickHouse usando `dlt`.

In [1]:
%pip install -r /home/jovyan/requirements.txt numpy

import os

import dlt
import pandas as pd
import pyarrow as pa
from dotenv import load_dotenv

load_dotenv('/home/jovyan/.env')

MINIO_ENDPOINT = os.environ.get('MINIO_ENDPOINT', 'http://fhbd-minio:9000')
MINIO_ACCESS_KEY = os.environ.get('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.environ.get('MINIO_SECRET_KEY', 'minioadmin123')
MINIO_BUCKET_ICEBERG = os.environ.get('MINIO_BUCKET_ICEBERG', 'iceberg')
AWS_REGION = os.environ.get('AWS_REGION', 'us-east-1')

NESSIE_URL = os.environ.get('NESSIE_URL', 'http://fhbd-nessie:19120/iceberg')
NESSIE_BRANCH = os.environ.get('NESSIE_BRANCH', 'main')
ICEBERG_NAMESPACE = os.environ.get('ICEBERG_NAMESPACE', 'nyc')
ICEBERG_TABLE = os.environ.get('ICEBERG_TABLE', 'yellow_tripdata')

CLICKHOUSE_HOST = os.environ.get('CLICKHOUSE_HOST', 'fhbd-clickhouse')
CLICKHOUSE_HTTP_PORT = int(os.environ.get('CLICKHOUSE_PORT', os.environ.get('CLICKHOUSE_HTTP_PORT', '8123')))
DEFAULT_NATIVE_PORT = '9002' if CLICKHOUSE_HOST in {'localhost', '127.0.0.1'} else '9000'
CLICKHOUSE_NATIVE_PORT = int(os.environ.get('CLICKHOUSE_NATIVE_PORT', DEFAULT_NATIVE_PORT))
CLICKHOUSE_USER = os.environ.get('CLICKHOUSE_USER', 'default')
CLICKHOUSE_PASSWORD = os.environ.get('CLICKHOUSE_PASSWORD', '')
CLICKHOUSE_DATABASE = os.environ.get('CLICKHOUSE_DATABASE', 'lakehouse')
CLICKHOUSE_TABLE = os.environ.get('CLICKHOUSE_TABLE', ICEBERG_TABLE)
CLICKHOUSE_WRITE_MODE = os.environ.get('CLICKHOUSE_WRITE_MODE', 'replace')


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyiceberg.catalog import load_catalog
import clickhouse_connect


def build_nessie_catalog():
    return load_catalog(
        'nessie',
        **{
            'type': 'rest',
            'uri': NESSIE_URL,
            'ref': NESSIE_BRANCH,
            'warehouse': f's3://{MINIO_BUCKET_ICEBERG}',
            's3.endpoint': MINIO_ENDPOINT,
            's3.access-key-id': MINIO_ACCESS_KEY,
            's3.secret-access-key': MINIO_SECRET_KEY,
            's3.path-style-access': 'true',
            's3.region': AWS_REGION,
        },
    )


def build_clickhouse_client(database=None):
    return clickhouse_connect.get_client(
        host=CLICKHOUSE_HOST,
        port=CLICKHOUSE_HTTP_PORT,
        username=CLICKHOUSE_USER,
        password=CLICKHOUSE_PASSWORD,
        database=database or CLICKHOUSE_DATABASE,
    )


def quote_identifier(identifier):
    return f'`{identifier.replace("`", "``")}`'


def get_dlt_clickhouse_credentials():
    return {
        'host': CLICKHOUSE_HOST,
        'port': CLICKHOUSE_NATIVE_PORT,
        'http_port': CLICKHOUSE_HTTP_PORT,
        'username': CLICKHOUSE_USER,
        'password': CLICKHOUSE_PASSWORD,
        'database': CLICKHOUSE_DATABASE,
        'secure': 0,
    }


In [3]:
catalog = build_nessie_catalog()
iceberg_table = catalog.load_table((ICEBERG_NAMESPACE, ICEBERG_TABLE))

eda_reader = iceberg_table.scan().to_arrow_batch_reader()
first_batch = next(iter(eda_reader))
sample_df = first_batch.to_pandas().head(10)
total_rows = first_batch.num_rows + sum(batch.num_rows for batch in eda_reader)
numeric_columns = [
    column
    for column in ['passenger_count', 'trip_distance', 'fare_amount', 'tip_amount', 'total_amount']
    if column in sample_df.columns
]

print('Iceberg table:', f'{ICEBERG_NAMESPACE}.{ICEBERG_TABLE}')
print('Total rows:', total_rows)
print('Columns:', list(first_batch.schema.names))
sample_df


Iceberg table: nyc.yellow_tripdata
Total rows: 3475226
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.60,1,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.50,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.60,1,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0
5,2,2025-01-01 00:48:24,2025-01-01 01:08:26,2,2.63,1,N,239,68,2,19.1,1.0,0.5,0.00,0.0,1.0,24.10,2.5,0.0,0.0
6,1,2025-01-01 00:14:47,2025-01-01 00:16:15,0,0.40,1,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0
7,1,2025-01-01 00:39:27,2025-01-01 00:51:51,0,1.60,1,N,234,148,1,12.1,3.5,0.5,2.00,0.0,1.0,19.10,2.5,0.0,0.0
8,1,2025-01-01 00:53:43,2025-01-01 01:13:23,0,2.80,1,N,148,170,1,19.1,3.5,0.5,3.00,0.0,1.0,27.10,2.5,0.0,0.0
9,2,2025-01-01 00:00:02,2025-01-01 00:09:36,1,1.71,1,N,237,262,2,11.4,1.0,0.5,0.00,0.0,1.0,16.40,2.5,0.0,0.0


In [4]:
if numeric_columns:
    sample_df[numeric_columns].describe().transpose()
else:
    print('No numeric columns available in the sample for EDA.')


In [5]:
@dlt.resource(name='iceberg_rows')
def iceberg_rows_resource():
    source_table = catalog.load_table((ICEBERG_NAMESPACE, ICEBERG_TABLE))
    for batch in source_table.scan().to_arrow_batch_reader():
        arrow_table = pa.Table.from_batches([batch])
        yield arrow_table.to_pylist()


bootstrap_client = build_clickhouse_client(database='default')
bootstrap_client.command(f'CREATE DATABASE IF NOT EXISTS {quote_identifier(CLICKHOUSE_DATABASE)}')
client = build_clickhouse_client()

pipeline = dlt.pipeline(
    pipeline_name='iceberg_to_clickhouse',
    destination=dlt.destinations.clickhouse(credentials=get_dlt_clickhouse_credentials()),
    progress='log',
)

resource = iceberg_rows_resource().with_name(CLICKHOUSE_TABLE)
resource.apply_hints(write_disposition=CLICKHOUSE_WRITE_MODE)

load_info = pipeline.run(resource)
load_info


------------------------ Extract iceberg_to_clickhouse -------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 884.16 MB (35.90%) | CPU usage: 0.00%

------------------------ Extract iceberg_to_clickhouse -------------------------
Resources: 0/1 (0.0%) | Time: 2.44s | Rate: 0.00/s
yellow_tripdata: 131072  | Time: 0.00s | Rate: 8589934592.00/s
Memory usage: 1263.30 MB (40.60%) | CPU usage: 0.00%

------------------------ Extract iceberg_to_clickhouse -------------------------
Resources: 0/1 (0.0%) | Time: 4.78s | Rate: 0.00/s
yellow_tripdata: 262144  | Time: 2.34s | Rate: 111836.48/s
Memory usage: 1244.14 MB (40.50%) | CPU usage: 0.00%

------------------------ Extract iceberg_to_clickhouse -------------------------
Resources: 0/1 (0.0%) | Time: 7.42s | Rate: 0.00/s
yellow_tripdata: 393216  | Time: 4.98s | Rate: 78965.49/s
Memory usage: 1243.30 MB (40.30%) | CPU usage: 0.00%

------------------------ Extract iceberg_to_clickhouse ----------------------

LoadInfo(pipeline=<dlt.pipeline(pipeline_name='iceberg_to_clickhouse', destination='clickhouse', default_schema_name='iceberg_to_clickhouse', schema_names=['iceberg_to_clickhouse'], first_run=False, dev_mode=False, is_active=True, pipelines_dir='/home/jovyan/.dlt/pipelines', working_dir='/home/jovyan/.dlt/pipelines/iceberg_to_clickhouse')>, metrics={'1773497881.5498369': [{'started_at': DateTime(2026, 3, 14, 14, 21, 27, 569862, tzinfo=Timezone('UTC')), 'finished_at': DateTime(2026, 3, 14, 14, 21, 36, 556870, tzinfo=Timezone('UTC')), 'job_metrics': {'yellow_tripdata.17f5897486.jsonl.gz': LoadJobMetrics(job_id='yellow_tripdata.17f5897486.jsonl.gz', file_path='/home/jovyan/.dlt/pipelines/iceberg_to_clickhouse/load/normalized/1773497881.5498369/started_jobs/yellow_tripdata.17f5897486.0.jsonl.gz', table_name='yellow_tripdata', started_at=DateTime(2026, 3, 14, 14, 21, 27, 851867, tzinfo=Timezone('UTC')), finished_at=DateTime(2026, 3, 14, 14, 21, 36, 472197, tzinfo=Timezone('UTC')), state='co

In [6]:
tables = [row[0] for row in client.query(f'SHOW TABLES FROM {quote_identifier(CLICKHOUSE_DATABASE)}').result_rows]
metadata_tables = [table for table in tables if table.startswith('_dlt_') or table == 'dlt_sentinel_table']
row_count = client.query(
    f'SELECT count() FROM {quote_identifier(CLICKHOUSE_DATABASE)}.{quote_identifier(CLICKHOUSE_TABLE)}'
).result_rows[0][0]

print('ClickHouse table:', f'{CLICKHOUSE_DATABASE}.{CLICKHOUSE_TABLE}')
print('Row count:', row_count)
print('Metadata tables:', metadata_tables)


ClickHouse table: lakehouse.yellow_tripdata
Row count: 3475226
Metadata tables: ['_dlt_loads', '_dlt_pipeline_state', '_dlt_version', 'dlt_sentinel_table']
